# Searching and Sorting Comparison Lab

```{contents}
:local:
:depth: 2
```

Use operation counts to support a purchasing report recommendation. Verify equivalent results before comparing costs. Clone input for each in-place sort and state whether counters include preparation or queries.

Download {download}`the complete search/sort implementations and reference checks <../../materials/22/SearchSortChecks.cs>` for a console project.


## Compare Equivalent Search Tasks

```{index} Compare Equivalent Search Tasks
```

Compare membership queries on the same sorted values. Linear search returns the first match and binary search any match; those index contracts differ, but both answer presence. Use empty, first, duplicate, last, and missing targets rather than one favorable query.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,200,300,400};
foreach(int target in new[]{100,200,400,250,500})
{
 int linear=ReportSearch.LinearFirst(sorted,target,EqualityComparer<int>.Default,out long lp);
 int binary=ReportSearch.BinaryAny(sorted,target,Comparer<int>.Default,out long bp);
 if((linear>=0)!=(binary>=0))throw new Exception("Presence mismatch.");
 Console.WriteLine($"target={target}, linear probes={lp}, binary probes={bp}, present={linear>=0}");
}
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


A binary result can identify a different duplicate occurrence. If the requirement is first sorted occurrence or an entire tie group, use the boundary interface from Section 22.1 and include its extra comparisons. Do not treat a sort’s new position as an original record index.

## Check Order and Occurrence Preservation

```{index} Check Order and Occurrence Preservation
```

Run all three elementary algorithms against an independent reference sequence. A sortedness check alone could accept dropped or duplicated values. Test empty, singleton, negative, duplicate, reverse, sorted, and extreme inputs.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[][] cases={Array.Empty<int>(),new[]{4},new[]{3,1,2},new[]{2,2,1,2},new[]{-3,-1,-2},new[]{1,2,3},new[]{3,2,1},new[]{int.MaxValue,int.MinValue}};
foreach(string algorithm in new[]{"selection","insertion","bubble"})
foreach(int[] values in cases)
{
 int[] copy=values.ToArray();
 if(algorithm=="selection")ElementarySort.Selection(copy,Comparer<int>.Default);
 else if(algorithm=="insertion")ElementarySort.Insertion(copy,Comparer<int>.Default);
 else ElementarySort.Bubble(copy,Comparer<int>.Default);
 if(!copy.SequenceEqual(values.OrderBy(x=>x)))throw new Exception("Sorted permutation failed.");
 Console.WriteLine($"{algorithm}, n={values.Length}: sorted permutation passed");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


Each algorithm receives the same initial data through a separate clone. Correctness is checked before a count is used as evidence. For stable algorithms, also compare labeled tie order against a stable reference; numeric output cannot reveal stability.

## Vary Input Order and Count Meaningfully

```{index} Vary Input Order and Count Meaningfully
```

Compare sorted, reverse, duplicate-heavy, and nearly sorted inputs. Key comparisons count comparer calls; swaps count exchanges; shifts count insertion moves to the right. These are distinct operations and should not be added without a stated cost model.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

(string name,int[] values)[] cases={
 ("sorted",new[]{1,2,3,4,5}), ("reverse",new[]{5,4,3,2,1}),
 ("ties",new[]{2,2,1,2,1}), ("near",new[]{1,2,4,3,5})};
foreach(var (name,values) in cases)
{
 var a=ElementarySort.Selection(values.ToArray(),Comparer<int>.Default);
 var b=ElementarySort.Insertion(values.ToArray(),Comparer<int>.Default);
 var d=ElementarySort.Bubble(values.ToArray(),Comparer<int>.Default);
 Console.WriteLine($"{name}: selection comparisons={a.Comparisons}, swaps={a.Swaps}; insertion comparisons={b.Comparisons}, shifts={b.Shifts}; bubble comparisons={d.Comparisons}, swaps={d.Swaps}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


Selection always makes ten key comparisons for n=5. Sorted insertion and optimized bubble need four; reverse insertion and shrinking-end bubble make ten. Input order matters for adaptive algorithms. A comparison counter does not measure allocation, memory locality, I/O, or elapsed time.

## Exact Formulas and Growth Models

```{index} Exact Formulas and Growth Models
```

Selection comparison count is exact, n(n−1)/2. The expression n log₂ n is a growth model rather than an exact merge-sort comparison count. Chapter 21 measured actual merge comparisons and writes; those quantities differ.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(long n in new[]{8L,32L,128L,512L})
 Console.WriteLine($"n={n}, selection comparisons={n*(n-1)/2}, merge growth model={n*Math.Log2(n):F0}");

| Algorithm variant | Best time | Worst time | Auxiliary space | Stable under shown policy? |
|---|---|---|---|---|
| Selection with suffix scan | Θ(n²) | Θ(n²) | Θ(1) | No |
| Insertion with strict shifts | Θ(n) | Θ(n²) | Θ(1) | Yes |
| Bubble with early exit/shrinking end | Θ(n) | Θ(n²) | Θ(1) | Yes |
| Range-buffer merge sort (Chapter 21) | Θ(n log n) | Θ(n log n) | Θ(n) | Yes |

Bounds describe growing nontrivial inputs under constant-cost comparisons. Library policies and implementation details are separate from these teaching variants.

## Include Preparation in Repeated Queries

```{index} Include Preparation in Repeated Queries
```

Compare repeated membership checks on an unsorted snapshot with sorting a copy once, then binary searching. Count library sort comparer calls as preparation; they can vary by runtime implementation. Query counts exclude copying, and the whole program still pays copying/allocation costs.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] original={900,100,700,300,500,200,800,400,600};
int[] queries={100,600,999,500,200,850,300,700,600,999,400,800};
long preparation=0,linearTotal=0,binaryTotal=0;
int[] sorted=original.ToArray();
Array.Sort(sorted,Comparer<int>.Create((a,b)=>{preparation++;return a.CompareTo(b);}));
foreach(int target in queries)
{
 int a=ReportSearch.LinearFirst(original,target,EqualityComparer<int>.Default,out long lp);
 int b=ReportSearch.BinaryAny(sorted,target,Comparer<int>.Default,out long bp);
 if((a>=0)!=(b>=0))throw new Exception("Workload result mismatch.");
 linearTotal+=lp;binaryTotal+=bp;
}
Console.WriteLine($"linear query probes={linearTotal}");
Console.WriteLine($"sort comparisons={preparation}, binary query probes={binaryTotal}, combined={preparation+binaryTotal}");
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


The general model compares qn with n log n+q log n, including constants and actual target distribution when making a practical decision. Small examples do not establish a universal query-count threshold. Frequent updates may require rebuilding the snapshot; exact-key lookup may favor a dictionary, while report ordering and range queries can favor a sorted index.

## Write a Requirement-Based Recommendation

```{index} Write a Requirement-Based Recommendation
```

State the requested result, current order, query frequency, update frequency, tie policy, memory/mutation limits, and key-comparison cost. Use measured counts for the tested workload and asymptotic bounds for scaling. Do not turn an arbitrary cutoff such as twenty items into a general rule.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string resultNeeded="all invoices with amount 200";
Console.WriteLine(resultNeeded);
Console.WriteLine("Static sorted report: lower/upper bounds locate the tie group; retain invoice IDs.");
Console.WriteLine("One unsorted query: scan once and collect all matching IDs.");
Console.WriteLine("Changing exact-key registry: consider dictionary lookup with an explicit duplicate-key policy.");

## Exercise

```{index} Exercise
```

Verify the exact selection comparison formula for sizes zero through six and show that sorted inputs still pay that cost.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes={0,1,2,3,4,5,6};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] sizes={0,1,2,3,4,5,6};
foreach(int n in sizes)
{
 int[] values=Enumerable.Range(0,n).ToArray();
 var counts=ElementarySort.Selection(values,Comparer<int>.Default);
 long expected=(long)n*(n-1)/2;
 if(counts.Comparisons!=expected)throw new Exception("Selection count mismatch.");
 Console.WriteLine($"n={n}, comparisons={counts.Comparisons}, expected={expected}");
}
public static class ElementarySort
{
    public sealed class Counts
    {
        public long Comparisons,Swaps,Shifts;
    }
    public static Counts Selection<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int start=0;start<values.Length-1;start++)
        {
            int smallest=start;
            for(int i=start+1;i<values.Length;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[smallest])<0)smallest=i;
            }
            if(smallest!=start){Swap(values,start,smallest);counts.Swaps++;}
        }
        return counts;
    }
    public static Counts Insertion<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int i=1;i<values.Length;i++)
        {
            T current=values[i];int j=i-1;
            while(j>=0)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[j],current)<=0)break;
                values[j+1]=values[j];counts.Shifts++;j--;
            }
            values[j+1]=current;
        }
        return counts;
    }
    public static Counts Bubble<T>(T[] values,IComparer<T> comparer)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        var counts=new Counts();
        for(int end=values.Length-1;end>0;end--)
        {
            bool swapped=false;
            for(int i=0;i<end;i++)
            {
                counts.Comparisons++;
                if(comparer.Compare(values[i],values[i+1])>0)
                {Swap(values,i,i+1);counts.Swaps++;swapped=true;}
            }
            if(!swapped)break;
        }
        return counts;
    }
    private static void Swap<T>(T[] values,int a,int b)
        => (values[a],values[b])=(values[b],values[a]);
}


n=0, comparisons=0, expected=0
n=1, comparisons=0, expected=0
n=2, comparisons=1, expected=1
n=3, comparisons=3, expected=3
n=4, comparisons=6, expected=6
n=5, comparisons=10, expected=10
n=6, comparisons=15, expected=15


```{rubric} Footnotes
```
[^1]: Timings require warmup, repeated runs, representative inputs, and a measured region excluding console output. Counts are useful evidence, but not timing substitutes.
[^2]: Preparation and maintenance are part of a workload cost. The best query algorithm can still be the wrong complete system choice.